# Usage Profiles: k-means Clustering

Segmentation of service users by how they engage with the service over time, using engagement features derived from a session log.

**All data in this notebook are simulated.** Feature distributions, sample size, the chosen number of clusters and the profile labels are invented for demonstration and do not correspond to any real dataset.

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

### Simulated session log and feature engineering

Each simulated user has a latent usage style (number of active months, spread, intensity). Their sessions are aggregated into monthly engagement features:

| Feature | Description |
| --- | --- |
| `active_months` | Distinct months with at least one session |
| `total_months` | Span in months between first and last session |
| `sessions_per_month` | Mean sessions in active months |
| `sessions_in_peak_month` | Maximum sessions in a single month |
| `gini_monthly` | Gini coefficient of the monthly distribution (unevenness of use) |

In [ ]:
rng = np.random.default_rng(11)
N_USERS = 5000

styles = rng.choice(4, N_USERS, p=[0.4, 0.25, 0.2, 0.15])
n_active = np.select([styles == 0, styles == 1, styles == 2, styles == 3],
                     [1, rng.integers(2, 4, N_USERS), rng.integers(3, 8, N_USERS), rng.integers(6, 13, N_USERS)])
spread = np.select([styles == 0, styles == 1, styles == 2, styles == 3],
                   [0, rng.integers(0, 3, N_USERS), rng.integers(2, 12, N_USERS), rng.integers(8, 24, N_USERS)])
intensity = np.select([styles == 0, styles == 1, styles == 2, styles == 3], [1.3, 3.0, 1.8, 1.5])

rows = []
for uid in range(N_USERS):
    months = np.sort(rng.choice(max(n_active[uid], n_active[uid] + spread[uid]), n_active[uid], replace=False))
    counts = 1 + rng.poisson(intensity[uid] - 1, n_active[uid]) if intensity[uid] > 1 else np.ones(n_active[uid], int)
    rows.append(pd.DataFrame({"user_id": uid, "month": months, "n_sessions": counts}))
monthly = pd.concat(rows, ignore_index=True)


def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n < 2 or x.sum() == 0:
        return np.nan
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())


features = monthly.groupby("user_id").agg(
    active_months=("month", "nunique"),
    total_months=("month", lambda m: m.max() - m.min() + 1),
    sessions_per_month=("n_sessions", "mean"),
    sessions_in_peak_month=("n_sessions", "max"),
    gini_monthly=("n_sessions", gini),
)
features.describe().round(2)

### Preprocessing

Missing values (e.g. the Gini index is undefined for a single active month) are imputed with the median, and features are standardised.

In [ ]:
FEATURES = list(features.columns)
df_imputed = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(features), columns=FEATURES)
X_scaled = StandardScaler().fit_transform(df_imputed)

### Model selection

k-means is fitted for k = 2 to 8 (the largest number still considered interpretable) and each solution is scored with inertia (elbow), silhouette, Calinski–Harabasz, Davies–Bouldin and a custom Dunn index.

In [ ]:
def dunn_index(X, labels):
    clusters = np.unique(labels)
    centroids = np.array([X[labels == c].mean(axis=0) for c in clusters])
    inter = cdist(centroids, centroids)
    np.fill_diagonal(inter, np.inf)
    diameters = [pdist(X[labels == c]).max() if (labels == c).sum() > 1 else 0 for c in clusters]
    return inter.min() / max(diameters) if max(diameters) > 0 else np.nan


rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "dunn": dunn_index(X_scaled, labels),
    })
results = pd.DataFrame(rows).set_index("k")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(results.index, results["inertia"], "o-")
axes[0].set(title="Elbow method", xlabel="k", ylabel="Inertia (SSE)")
axes[1].plot(results.index, results["silhouette"], "o-", color="tab:red")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
plt.show()

results.round(3)

The indices are weighed together, giving priority to the silhouette and Calinski–Harabasz scores. Here the choice is automated with a simple rank average:

In [ ]:
ranks = pd.concat([
    results["silhouette"].rank(ascending=False),
    results["calinski_harabasz"].rank(ascending=False),
    results["davies_bouldin"].rank(),
    results["dunn"].rank(ascending=False),
], axis=1)
weights = np.array([2, 2, 1, 1])
best_k = int((ranks * weights).sum(axis=1).idxmin())
print("Selected k:", best_k)

### Cluster description

In [ ]:
final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_imputed["cluster"] = final.fit_predict(X_scaled)

summary = df_imputed.groupby("cluster")[FEATURES].mean()
summary["n_users"] = df_imputed["cluster"].value_counts().sort_index()
summary.round(2)

Each cluster gets a descriptive label from its feature profile (a simple rule here, standing in for a manual or LLM-assisted naming step):

In [ ]:
def label_cluster(row):
    if row["active_months"] < 1.5:
        return "One-off"
    if row["sessions_per_month"] > 2.5:
        return "Intensive short-term"
    if row["total_months"] > 12:
        return "Long-term"
    return "Regular moderate" if row["gini_monthly"] < 0.15 else "Irregular moderate"


names = summary.apply(label_cluster, axis=1)
names = names + names.groupby(names).cumcount().map(lambda i: "" if i == 0 else f" ({i + 1})")
df_imputed["profile"] = df_imputed["cluster"].map(names)
names

In [ ]:
pcs = PCA(n_components=2).fit_transform(X_scaled)
plot_df = pd.DataFrame(pcs, columns=["PC1", "PC2"]).assign(profile=df_imputed["profile"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=plot_df.sample(2000, random_state=0), x="PC1", y="PC2", hue="profile",
                palette="tab10", s=12, ax=axes[0])
axes[0].set_title("Clusters in PCA space")

order = df_imputed["profile"].value_counts().index
sns.countplot(data=df_imputed, x="profile", order=order, hue="profile", palette="tab10", legend=False, ax=axes[1])
axes[1].set_title("Users per profile")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()